# W4C2 Lab: Tensors, autograd, and a line through real data

Run every cell from the top. **Everything already works.**

Parts 1 to 3 we run together, cell by cell; after the break it is yours. Each part ends with a **TRY IT**: one question, one empty cell.

Today you will:

1. Meet the **tensor**: a NumPy array with two extra powers.
2. Watch **autograd** work out which way is downhill.
3. Fit a **straight line** through real data, and check it against the formula.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
MAROON = "#7C2529"
GRAY = "#8e8e93"

# Seeding makes every random number in this notebook the same on every machine,
# so the numbers you see are the numbers in the comments.
torch.manual_seed(0)

print("torch", torch.__version__)

## Part 1. A tensor is an array that can learn


PyTorch calls a NumPy array a **tensor**. Shape, axis and broadcasting are
unchanged. A tensor adds two things:

1. it can run on a GPU, and
2. it **remembers how it was computed**, which is what makes training possible.

Part 1 is the first. Part 2 is the second.


<img src="images/tensor-vs-array.png" width="640">

In [ ]:
# Three ways to make a tensor. Print the shape every time.
from_list = torch.tensor([[1.0, 2.0, 3.0],
                          [4.0, 5.0, 6.0]])
from_numpy = torch.tensor(np.array([[1, 2, 3], [4, 5, 6]]))
zeros = torch.zeros(2, 3)

print("from_list  shape", tuple(from_list.shape), "dtype", from_list.dtype)
print("from_numpy shape", tuple(from_numpy.shape), "dtype", from_numpy.dtype)
print("zeros      shape", tuple(zeros.shape), "dtype", zeros.dtype)
print()
print(from_list)

In [ ]:
# DTYPE is the one place PyTorch is stricter than NumPy, and it bites everyone
# once. Networks do their arithmetic in float32. Class labels stay int64.
counts = torch.tensor([[2, 0, 1], [0, 3, 1]])          # whole numbers -> int64
features = counts.float()                              # -> float32

print("counts  ", counts.dtype)
print("features", features.dtype)
print()
print("Layers want float32. The error you get for int64 never says 'dtype'.")

In [ ]:
# SHAPE, again. reshape rearranges the same numbers into a new shape.
row = torch.arange(12.0)                # 12 numbers in a line
print("row       ", tuple(row.shape))

grid = row.reshape(3, 4)                # 3 rows of 4
print("grid      ", tuple(grid.shape))

tall = row.reshape(4, -1)               # -1 means "work the rest out for me"
print("tall      ", tuple(tall.shape))

# The one you will actually need: adding a batch dimension of 1.
one_example = torch.tensor([1.0, 2.0, 3.0])
batched = one_example.reshape(1, -1)
print()
print("one_example", tuple(one_example.shape), "-> batched", tuple(batched.shape))
print("A layer wants (batch, features), even for a batch of one.")

In [ ]:
# MATH. * is elementwise. @ is matrix multiplication, and it is the operation
# every neural network is built from.
a = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
b = torch.tensor([[10.0, 20.0],
                  [30.0, 40.0]])

print("a * b  (elementwise):")
print(a * b)
print()
print("a @ b  (matrix multiply):")
print(a @ b)
print()

# The shape rule for @ is worth memorising:  (n, k) @ (k, m) -> (n, m)
documents = torch.zeros(180, 221)     # 180 documents, 221 word counts each
weights = torch.zeros(221, 8)         # 221 word counts in, 8 numbers out
print("(180, 221) @ (221, 8) ->", tuple((documents @ weights).shape))
print("The inner 221s cancel. That is the whole rule.")

In [ ]:
# ================== TRY IT 1 ==================
# `row` is twelve numbers in a line, shape (12,). A layer wants
# (batch, features). Turn it into a batch of one.
# ==============================================


## Part 2. Autograd: which way is downhill?


`requires_grad=True` starts a recording. `.backward()` replays it backwards and
fills in `.grad` on every input: the slope of the result with respect to that
number.

That is the whole mechanism behind training.


<img src="images/autograd-graph.png" width="640">

In [ ]:
# The smallest possible example.  y = x^2, so dy/dx = 2x, and at x = 3 that is 6.
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

y.backward()          # walk the recording backwards

print("x      =", x.item())
print("y      =", y.item())
print("x.grad =", x.grad.item(), "  <- dy/dx = 2x = 6")
print()
print("If x grows a little, y grows about 6 times as much. To shrink y, move x")
print("the other way.")

In [ ]:
# Now use it. One weight, one target, twenty steps downhill.
w = torch.tensor([5.0], requires_grad=True)   # start far from the answer
target = torch.tensor([2.0])                  # the value we want w to reach

learning_rate = 0.1
history = []

for step in range(20):
    loss = (w - target) ** 2      # 1. how wrong are we?
    loss.backward()               # 2. which way is downhill?
    with torch.no_grad():         # 3. step, without recording the step itself
        w -= learning_rate * w.grad
    w.grad.zero_()                # 4. clear the gradient for the next round
    history.append(loss.item())

print(f"w started at 5.000 and ended at {w.item():.3f}, target was 2.0")
print(f"loss fell from {history[0]:.3f} to {history[-1]:.5f}")

plt.figure(figsize=(6, 3))
plt.plot(history, marker="o", color=MAROON)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Twenty steps downhill")
plt.show()


**loss, backward, step, zero.** Every model in this course is trained by those
four lines, over millions of weights instead of one.

<img src="images/training-loop.png" width="640">


In [ ]:
# ================== TRY IT 2 ==================
# Set x to 4.0 with `requires_grad=True`, compute y = x ** 3, and read
# `x.grad`. What should it be by hand?
# ==============================================


## Part 3. A line through real data


The simplest model there is: `y = wx + b`, **two** numbers to learn.

The data is 240 movie reviews. How many characters does one more word buy you?


<img src="images/linear-model.png" width="640">

In [ ]:
import pandas as pd

reviews = pd.read_csv("data/reviews.csv")
reviews["n_words"] = reviews["text"].str.split().str.len()
reviews["n_chars"] = reviews["text"].str.len()

plt.figure(figsize=(6, 3.6))
plt.scatter(reviews["n_words"], reviews["n_chars"], s=22, color=GRAY, alpha=0.6)
plt.xlabel("words in the review")
plt.ylabel("characters")
plt.title("240 reviews")
plt.show()

print("There is clearly a line in there. Our job is to find it.")

In [ ]:
# The data, as tensors. Note the shape: a COLUMN, not a flat row.
X = torch.tensor(reviews["n_words"].to_numpy(), dtype=torch.float32).reshape(-1, 1)
Y = torch.tensor(reviews["n_chars"].to_numpy(), dtype=torch.float32).reshape(-1, 1)

print("X", tuple(X.shape), X.dtype)
print("Y", tuple(Y.shape), Y.dtype)
print()
print("(240, 1) is 240 examples with one feature each. The second axis stays,")
print("even with only one number in it.")

In [ ]:
# THE MODEL. One input, one output, so one weight and one bias.
model = nn.Linear(1, 1)

print(model)
print()
print("before training, the two numbers are random:")
print(f"   w = {model.weight.item():.3f}")
print(f"   b = {model.bias.item():.3f}")
print()
print("so the line it draws right now is nonsense:")
print(f"   a 20-word review would have {model.weight.item() * 20 + model.bias.item():.1f} characters")

In [ ]:
# TRAINING. The same four lines as Part 2, now over two numbers instead of one.
EPOCHS = 2000
LEARNING_RATE = 1.0

torch.manual_seed(0)
model = nn.Linear(1, 1)
loss_function = nn.MSELoss()          # square every gap, take the mean
optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = []
for epoch in range(EPOCHS):
    optimiser.zero_grad()             # 1. clear
    predicted = model(X)              # 2. measure
    loss = loss_function(predicted, Y)
    loss.backward()                   # 3. blame
    optimiser.step()                  # 4. step
    history.append(loss.item())

w = model.weight.item()
b = model.bias.item()

print(f"loss fell from {history[0]:.1f} to {history[-1]:.3f}")
print()
print(f"   chars = {w:.3f} x words + {b:.3f}")

In [ ]:
# LOOK AT IT. The line it learned, and the loss on the way there.
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].scatter(reviews["n_words"], reviews["n_chars"], s=20, color=GRAY, alpha=0.55)
line_x = np.array([reviews["n_words"].min(), reviews["n_words"].max()])
axes[0].plot(line_x, w * line_x + b, color=MAROON, linewidth=2.5)
axes[0].set_xlabel("words")
axes[0].set_ylabel("characters")
axes[0].set_title(f"chars = {w:.2f} x words + {b:.1f}")

axes[1].plot(history, color=MAROON)
axes[1].set_yscale("log")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mean squared error")
axes[1].set_title("Loss")

plt.tight_layout()
plt.show()

In [ ]:
# IS IT RIGHT? There is a formula for the best straight line, and NumPy has it.
# Gradient descent should land on exactly the same answer.
best_w, best_b = np.polyfit(reviews["n_words"], reviews["n_chars"], 1)

print(f"gradient descent : w = {w:.3f}   b = {b:.3f}")
print(f"the formula      : w = {best_w:.3f}   b = {best_b:.3f}")
print()
print("Identical. A line has a formula, so you never needed the loop. After this")
print("week there is no formula, and the loop is all you have.")

In [ ]:
# ================== TRY IT 3 ==================
# How many characters would you expect in a 30-word review?
# And for a review of zero words. Does that second number mean anything?
# ==============================================



---

## Your turn

Two tasks. Each changes one thing and redraws the line.


In [ ]:
# GIVEN, so both tasks are one line each.
def fit_line(x_column, y_column, epochs=2000, learning_rate=1.0, seed=0):
    """Fit y = wx + b by gradient descent. Returns w, b and the loss history."""
    x = torch.tensor(reviews[x_column].to_numpy(), dtype=torch.float32).reshape(-1, 1)
    y = torch.tensor(reviews[y_column].to_numpy(), dtype=torch.float32).reshape(-1, 1)

    torch.manual_seed(seed)
    line = nn.Linear(1, 1)
    loss_function = nn.MSELoss()
    optimiser = torch.optim.Adam(line.parameters(), lr=learning_rate)

    losses = []
    for epoch in range(epochs):
        optimiser.zero_grad()
        loss = loss_function(line(x), y)
        loss.backward()
        optimiser.step()
        losses.append(loss.item())
    return line.weight.item(), line.bias.item(), losses


def show_fit(x_column, y_column, w, b, losses):
    """Scatter the two columns, draw the fitted line, and report the loss."""
    plt.figure(figsize=(6, 3.6))
    plt.scatter(reviews[x_column], reviews[y_column], s=20, color=GRAY, alpha=0.55)
    edge = np.array([reviews[x_column].min(), reviews[x_column].max()])
    plt.plot(edge, w * edge + b, color=MAROON, linewidth=2.5)
    plt.xlabel(x_column)
    plt.ylabel(y_column)
    plt.title(f"{y_column} = {w:.3f} x {x_column} + {b:.3f}")
    plt.show()
    print(f"final loss: {losses[-1]:.3f}")


print("fit_line() and show_fit() are ready to use.")

In [ ]:
# ================== YOUR TURN 1 ==================
# Two thousand epochs is a lot for two numbers. How few can you get
# away with? Try 200, then 500, then 1000, and watch both the line and
# the loss.
#
# Expected: 200 epochs gives w = 4.876, b = 9.210 and a loss of 51.730: the
#           slope is too steep and the line sits low on the left. 500 gets to
#           49.465, 1000 to 49.203, and 2000 to 49.202, which is the best any
#           straight line can do. The slope converges long before the intercept
#           does.
# =================================================
EPOCHS = 2000          # <-- change me

w1, b1, losses1 = fit_line("n_words", "n_chars", epochs=EPOCHS)
show_fit("n_words", "n_chars", w1, b1, losses1)

In [ ]:
# ================== YOUR TURN 2 ==================
# Now predict `n_chars` from `rating` instead. Change the x column,
# compare the final loss with 49.202, and decide whether this line
# means anything.
#
# Expected: w = 0.698, b = 93.772, and a loss of 277.695, more than five times
#           worse. Predicting the average character count for every review, and
#           ignoring the rating entirely, would give 281.58. The line is barely
#           better than no model at all. It still drew you a line: linear
#           regression always does, whether or not there is anything there.
# =================================================
w2, b2, losses2 = fit_line("n_words", "n_chars")     # <-- change the x column

show_fit("n_words", "n_chars", w2, b2, losses2)      # <-- and here
print("for comparison, predicting the mean every time would score",
      round(reviews["n_chars"].var(ddof=0), 2))

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   row.reshape(1, -1)
#   torch.Size([1, 12]). reshape(1, 12) works too; -1 means "you work it out".

# TRY IT 2
#   x = torch.tensor(4.0, requires_grad=True)
#   y = x ** 3
#   y.backward()
#   x.grad  ->  48.0, which is 3 * 4**2.

# TRY IT 3
#   w * 30 + b   ->  150.4 characters
#   b on its own ->  17.7 characters for a review with no words at all.
#   The shortest review in the data is 11 words, so the line has never been
#   anywhere near x = 0 and has no business being asked about it. An intercept
#   is a number that positions the line, not a measurement.

# YOUR TURN 1
#   EPOCHS = 200    w = 4.876  b =  9.210  loss 51.730
#   EPOCHS = 500    w = 4.569  b = 14.948  loss 49.465
#   EPOCHS = 1000   w = 4.429  b = 17.555  loss 49.203
#   EPOCHS = 2000   w = 4.423  b = 17.668  loss 49.202
#
#   The slope settles first and the intercept crawls after it. That is because
#   the reviews are 11 to 26 words long, so nudging b moves the line much less
#   than nudging w does, and the gradient for b is correspondingly small.

# YOUR TURN 2
#   w2, b2, losses2 = fit_line("rating", "n_chars")
#   show_fit("rating", "n_chars", w2, b2, losses2)
#
#   w = 0.698, b = 93.772, loss 277.695 against 49.202 for word count.
#   Predicting the mean every time scores 281.58, so this line is worth almost
#   nothing. Positive reviews are a little longer than negative ones, and that
#   is the whole of the signal.
#
#   The lesson: a fitted line is not evidence of a relationship. Always compare
#   the loss against the do-nothing baseline.

# The three things worth carrying out of today:
#   1. A tensor is a NumPy array that records how it was built, so it can be
#      differentiated. requires_grad is the switch.
#   2. loss, backward, step, zero. Every training loop in this course is those
#      four lines, whether the model has two numbers or two hundred billion.
#   3. A model always returns an answer. Whether the answer is worth anything is
#      a separate question, and the loss against a baseline is how you ask it.